# One hot encoded dataset

Mismo procedimiento que antes pero codificamos más tarde 

In [15]:
import pandas as pd
df_prep = pd.read_csv("sequences_df_prep_EN.csv")
df = df_prep.copy()

In [ ]:
from sklearn.preprocessing import LabelEncoder

df = df_prep.copy()

le_mix = LabelEncoder()
encoded_mix = le_mix.fit_transform(df_prep['mixture'])
df['encoded_mixture']=encoded_mix

le_container = LabelEncoder()
encoded_container = le_container.fit_transform(df_prep['container'])
df['encoded_container']=encoded_container

In [16]:
hora = pd.to_datetime(df_prep['initepoch'], unit='ms')
df['hora_decimal'] = hora.dt.hour + hora.dt.minute / 60 + hora.dt.second / 3600

In [ ]:
df = df[['user','initdayofweek', 'hora_decimal', 'shift', 'encoded_mixture', 'additive','encoded_container']]

In [18]:
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.metadata import Metadata

metadata = Metadata.detect_from_dataframe(data=df, table_name='interactions')

synthesizer = GaussianCopulaSynthesizer(metadata)
synthesizer.fit(df)

synthetic_data = synthesizer.sample(num_rows=100)

c:\Users\manex\miniconda3\envs\programacion\Lib\site-packages\sdv\single_table\base.py:123: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


In [19]:
synthetic_data['user'] = "synthetic_1"

synthetic_data.loc[synthetic_data['hora_decimal'] < 12.0, 'mixture'] = "mixture2"
synthetic_data.loc[synthetic_data['hora_decimal'] < 12.0, 'additive'] = 3

synthetic_data.loc[synthetic_data['hora_decimal'] > 12.0, 'mixture'] = 'mixture3'
synthetic_data.loc[synthetic_data['hora_decimal'] > 12.0, 'additive'] = 0

synthetic_data['container'] = 'yes'

In [20]:
synthetic_data_2 = synthesizer.sample(num_rows=100)
synthetic_data_2['user'] = 'synthetic_2'

synthetic_data_2.loc[synthetic_data_2['initdayofweek'] <= 3 , 'mixture'] = 'mixture8'
synthetic_data_2.loc[synthetic_data_2['initdayofweek'] > 3, 'mixture'] = 'mixture9'
synthetic_data_2['additive'] = 0
synthetic_data_2['container'] = 'yes'

In [21]:
synthetic_data_3 = synthesizer.sample(num_rows=100)
synthetic_data_3['user'] = 'synthetic_3'
synthetic_data_3['mixture'] = 'mixture6'
synthetic_data_3['additive'] = 0
synthetic_data_3['container'] = 'no'

synthetic_data_4 = synthesizer.sample(num_rows=100)
synthetic_data_4['user'] = 'synthetic_4'
synthetic_data_4['mixture'] = 'mixture7'
synthetic_data_4['additive'] = 3
synthetic_data_4['container'] = 'yes'

In [22]:
df = pd.concat([df, synthetic_data, synthetic_data_2, synthetic_data_3, synthetic_data_4], ignore_index=True)

In [23]:
df = df.sample(frac=1).reset_index(drop=True)

In [29]:
encoded_user = pd.get_dummies(df['user'], prefix_sep='_')

In [30]:
df_encoded = pd.concat([df, encoded_user], axis=1)

In [31]:
df_encoded.to_csv("encoded_cooked_df.csv", index=False)

In [ ]:
from sklearn.model_selection import train_test_split

cooked_train, cooked_test = train_test_split(df, test_size
                                             =0.2, random_state=42)

cooked_train.to_csv("encoded_cooked_train.csv", index=False)
cooked_test.to_csv("encoded_cooked_test.csv", index=False)

: 